# 1、将原始keywords_train/test.jsonl做格式转换

In [ ]:
from datasets import load_dataset
# 1.1 加载数据
data = load_dataset("json",data_files={"train":"./data/keywords_data_train.jsonl","test":"./data/keywords_data_test.jsonl"})

In [ ]:
# 1.2 将数据转换成SFTTrainer所需要的 Language Modeling 这种类型，对话格式的数据
def convert_func(examples:dict[str, list]):
    """
    接收的参数，就是.map方法传递的，原始的数据，以批次形式接收
    """
    conversation_lists: list[list] =examples["conversation"]
    messages_lists:list[list] = []
    for conversation in conversation_lists:
        # conversation是单条样本所对应的列表：
        human_message = conversation[0]["human"]
        assistant_message = conversation[0]["assistant"]
        message_list = [ 
            {"role":"user","content":human_message},
            {"role":"assistant","content":assistant_message}
        ]
        messages_lists.append(message_list)

    return {"messages":messages_lists}



converted_data = data.map(convert_func,batched=True,remove_columns=data["train"].column_names,)

# 2、构造SFTConfig对象（需要理解SFTConfig有哪些重点参数）

In [ ]:
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs/04_trl_sft_demo"
config = SFTConfig(
    # 数据规模相关的
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps= 32,
    max_steps=500,
    # num_train_epochs= # max_steps会比num_train_epochs的优先级更高
    # 训练可视化相关
    logging_strategy="steps",
    logging_steps=25,
    report_to="tensorboard", # 要想去进一步制定tensorboard 日志文件保存位置，需要通过os.environ去指定,
    # 学习率和优化器相关
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps= 0.1,
    # optim="" 优化器的类型，默认值就是adamW
    # 评估和保存相关
    eval_strategy="steps",
    eval_steps=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    output_dir="./finetuned/04_trl_sft_demo", # 保存的是检查点
    bf16=True,
    gradient_checkpointing=False,
    activation_offloading=False,
    max_length=700,
    # 原生的qwen3的聊天模板，和assistant_only_loss参数不兼容，所以需要基于原生的chat_template进行修改，得到new_chat_template.jinja文件
    # 可以通过chat_template_path传递新的chat_template文件
    assistant_only_loss=True,
    chat_template_path="./new_chat_template.jinja"
)

# 3、构造LoRAConfig,通过get_peft_model，传入原始模型和LoraConfig，获取一个PEFTModel

In [2]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
origin_model = AutoModelForCausalLM.from_pretrained("./model/Qwen3-0.6B/")
model = AutoModelForCausalLM.from_pretrained("./model/Qwen3-0.6B/")
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    # target_modules=["q_proj","v_proj"]
    # target_modules=["q_proj","k_proj","v_proj"]
    target_modules="all-linear", # 表示的就是在attention和FFN当中的所有参数矩阵，插入LoRA,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model,lora_config)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1295.30it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1316.14it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [14]:
origin_model.model.layers[0].self_attn.q_proj.weight.requires_grad

True

In [13]:
peft_model.base_model.model.model.layers[0].self_attn.q_proj.lora_A["default"].weight

Parameter containing:
tensor([[ 0.0226,  0.0267,  0.0142,  ...,  0.0211, -0.0165,  0.0028],
        [ 0.0150,  0.0053, -0.0272,  ...,  0.0228, -0.0200,  0.0128],
        [-0.0019, -0.0073, -0.0028,  ...,  0.0269,  0.0082, -0.0250],
        ...,
        [-0.0216, -0.0110,  0.0217,  ..., -0.0084, -0.0273, -0.0157],
        [-0.0064,  0.0096,  0.0161,  ...,  0.0154,  0.0036, -0.0126],
        [ 0.0197,  0.0101,  0.0003,  ...,  0.0135, -0.0289,  0.0295]],
       requires_grad=True)

# 4、基于SFTConfig，数据集，模型(peft_model)，tokenizer等，去构建一个SFTTrainer实例

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.trainer.sft_trainer import SFTTrainer
from tyro import conf
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")

trainer = SFTTrainer(
    model=peft_model, # 此处传递的，不再是原模型，而是通过get_peft_model所得到的新模型
    args=config,
    train_dataset=converted_data["train"],
    eval_dataset=converted_data["test"],
    # processing_class指的就是tokenizer参数
    processing_class= tokenizer
)

# 5、训练，保存

In [ ]:
trainer.train()
trainer.save_model("./finetuned/05_trl_peft_demo")